# Exp 20 — Bounded single agent pilot (always run, development, group-C claims)
**Question:** even though Exp 19 found no case in the 13 group-C claims genuinely needs one, does a real, model-directed ReAct-style agent do as well as Exp 18's fixed workflow, on the same claims? Run regardless of the audit's conclusion, as agreed, so the final call rests on measured data.

**Agent design:** given the claim, the frozen M4 RAG excerpts (Exp 9), and the list of typed tools (Exp 17) with descriptions -- but **not** the pre-resolved facts from Exp 12, since the point is to see whether the model can decide *what to fetch* on its own. Loop: the model emits one tool call or a final decision each turn; the observation is fed back; repeat.
**Guardrails:** step cap 6, action de-duplication (repeated identical calls served from cache, not counted against the cap), read-only tools only, explicit REQUEST_INFORMATION/ESCALATE as valid stops, stops as soon as it has enough evidence.
**Logging:** tool, args, observation, turn count, tokens, latency, cost, final decision -- no hidden chain-of-thought stored.
**Data:** the same 13 group-C claims as Exp 18/19, for a matched comparison.

In [1]:
import json, os, re, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parents[1]; sys.path.insert(0, str(ROOT))
from src import config as C, llm, llm_exp, retrieval, retrievers, embed, evaluate, tools
C.load_env()
pd.set_option('display.width', 260, 'display.max_columns', 40, 'display.max_colwidth', 100, 'display.max_rows', 200)
EXP = 'EXP20_AGENT'; EMB = 'voyageai/voyage-4-lite'; CFG = 'fixed600_100'; K = 8; MAX_STEPS = 6
gt = evaluate.load_gt(); cases = {c['case_id']: c for c in llm_exp.cases_for('DEVELOPMENT')}
groupC = [gt[cid] for cid in gt if gt[cid]['split']=='DEVELOPMENT' and gt[cid]['architecture_group']=='C_AGENT_DYNAMIC']
targets = [cases[g['case_id']] for g in groupC]
print(len(targets), 'group-C claims | spent so far $%.4f of $%s' % (llm.spent(), os.environ['MAX_BUDGET_USD']))

13 group-C claims | spent so far $2.0597 of $3.50


## Frozen RAG context (Exp 9's M4 config, reused) and the tool catalogue for the agent's system prompt

In [2]:
R = retrievers.Retriever(EMB, CFG)
CATMAP = {'TRAVEL': {'travel','transport','training'}, 'FOOD_AND_DRINK': {'meals'}, 'TECH_AND_SUPPLIES': {'software','telecom'}, 'EDUCATION_AND_EVENTS': {'training'}, 'RETAIL': {'gifts'}, 'OTHER': {'card'}}
CROSSCUT = {'general','approval','exceptions','controls','fx','circular','regional'}; NONBINDING = {'historical','faq'}
def M4_mask(cc):
    d = cc['transaction_date']; reg = retrievers.REGION.get(cc['bill']['country'], ''); allowed_cat = CROSSCUT | CATMAP.get(cc['bill']['merchant_category'], set())
    return np.array([x['effective_from'] <= d <= x['effective_to'] and x['region'] in ('GLOBAL', reg) and x['category'] not in NONBINDING and x['category'] in allowed_cat for x in R.chunks])
qv = embed.embed(EMB, [retrieval.claim_query(cc) for cc in targets], tag=EXP)
CTX = {cc['case_id']: R.context(R.rank('dense', 'unused', q, K, M4_mask(cc))) for cc, q in zip(targets, qv)}
TOOL_CATALOGUE = '\n'.join(f"- {name}({', '.join(spec)}): {desc}" for name, (fn, spec, desc) in tools.TOOLS.items())
print(TOOL_CATALOGUE[:600])

- get_employee_profile(employee_id): Employee grade, region, department and status by employee_id. Use for grade-dependent limits.
- get_travel_request(employee_id, date): The employee's travel request covering a date (status, destination, and any linked event_id or exception_id). Use to establish trip approval.
- get_manager_approval(expense_id): Raw manager-approval records attached to an expense_id, with type, status and dates. Not for policy exceptions. For a yes/no answer on validity, use validate_approval instead of interpreting this yourself.
- get_exception_record(exception_id): An alr


/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: divide by zero encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))
/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: overflow encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))
/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: invalid value encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))


## The bounded agent loop

In [3]:
SYSTEM = ('''You are ExpenseGuard, a finance assistant deciding whether an employee expense claim is ready for reimbursement. You may call read-only tools to look up enterprise records you need. You do not have every record up front: decide what to fetch.
Guardrails: you may take at most {max_steps} steps. Do not repeat an identical tool call. Tools are read-only: never ask for a write/payment action. Stop as soon as you have enough evidence -- do not call tools you do not need.
At each turn, return ONLY one JSON object, either a tool call: {{"action": "call_tool", "tool": "<name>", "args": {{...}}}} , or your final answer: {{"action": "final", "decision": "APPROVE|REJECT|REQUEST_INFORMATION|ESCALATE", "policy_evidence": [...], "missing_fields": [...], "explanation": "<=2 sentences"}}.
Available tools:
{tools}
''').format(max_steps=MAX_STEPS, tools=TOOL_CATALOGUE)

def run_agent(cc):
    ctx = CTX[cc['case_id']]
    messages = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': f"CLAIM:\n{json.dumps(llm_exp.visible(cc), indent=1)}\n\nRETRIEVED POLICY EXCERPTS:\n{ctx}"}]
    seen = {}; trace = []; cost = 0.0; wrong_tool = 0; step_cap_hit = False
    for step in range(MAX_STEPS):
        r = llm.chat(os.environ['PAID_MODEL'], messages, temperature=0, max_tokens=350, tag=EXP, case_id=cc['case_id'])
        cost += r['cost_usd']
        try:
            d = json.loads(r['text'])
        except Exception:
            d = {'action': 'final', 'decision': 'ESCALATE', 'policy_evidence': [], 'missing_fields': [], 'explanation': 'Could not parse a structured response.'}
        if d.get('action') == 'final':
            return {'decision': d.get('decision'), 'policy_evidence': d.get('policy_evidence', []), 'missing_fields': d.get('missing_fields', []), 'explanation': d.get('explanation'),
                    'trace': trace, 'turns': step + 1, 'cost_usd': round(cost, 6), 'wrong_tool_calls': wrong_tool, 'step_cap_hit': False}
        name, args = d.get('tool'), d.get('args') or {}
        key = (name, tuple(sorted((k, tuple(v) if isinstance(v, list) else v) for k, v in args.items())))
        if key in seen:
            obs = seen[key]  # de-duplicated: served from cache, still shown to the model but not a new lookup
        else:
            obs = tools.call_tool(name, args) if name in tools.TOOLS else {'ok': False, 'found': False, 'data': None, 'error': f'unknown tool: {name}'}
            seen[key] = obs
            if not obs['ok'] or not obs['found']:
                wrong_tool += 1
        trace.append({'tool': name, 'args': args, 'observation': obs})
        messages.append({'role': 'assistant', 'content': r['text']})
        messages.append({'role': 'user', 'content': f'OBSERVATION: {json.dumps(obs, default=str)}'})
    return {'decision': 'ESCALATE', 'policy_evidence': [], 'missing_fields': [], 'explanation': f'Step cap ({MAX_STEPS}) reached; escalated instead of guessing.',
            'trace': trace, 'turns': MAX_STEPS, 'cost_usd': round(cost, 6), 'wrong_tool_calls': wrong_tool, 'step_cap_hit': True}

results = {cc['case_id']: run_agent(cc) for cc in targets}
print('spent so far $%.4f of $%s' % (llm.spent(), os.environ['MAX_BUDGET_USD']))
for cid, r in results.items(): print(cid, r['decision'], '| turns', r['turns'], '| step_cap_hit', r['step_cap_hit'], '| cost $%.4f' % r['cost_usd'])

spent so far $2.0933 of $3.50
X2-005 REJECT | turns 6 | step_cap_hit False | cost $0.0038
X2-018 REJECT | turns 6 | step_cap_hit False | cost $0.0032
X2-032 REJECT | turns 5 | step_cap_hit False | cost $0.0032
X2-037 APPROVE | turns 6 | step_cap_hit False | cost $0.0030
X2-047 APPROVE | turns 5 | step_cap_hit False | cost $0.0031
X2-055 APPROVE | turns 6 | step_cap_hit False | cost $0.0032
X2-087 APPROVE | turns 4 | step_cap_hit False | cost $0.0023
X2-089 APPROVE | turns 6 | step_cap_hit False | cost $0.0033
X2-095 REJECT | turns 6 | step_cap_hit False | cost $0.0037
X2-104 APPROVE | turns 6 | step_cap_hit False | cost $0.0038
X2-128 ESCALATE | turns 6 | step_cap_hit True | cost $0.0037
X2-142 ESCALATE | turns 6 | step_cap_hit True | cost $0.0033
X2-145 ESCALATE | turns 6 | step_cap_hit True | cost $0.0036


## Save results and compare against Exp 18 (fixed workflow) on the identical 13 claims

In [4]:
recs = [{'case_id': cid, 'predicted_decision': r['decision'], 'policy_evidence': r['policy_evidence'], 'missing_fields': r['missing_fields'], 'manual_review_required': r['decision']=='ESCALATE',
         'latency_ms': 0, 'input_tokens': 0, 'output_tokens': 0, 'model_cost_usd': r['cost_usd'], 'error': None} for cid, r in results.items()]
sA, rowsA = evaluate.evaluate(recs, gt)
out = C.RESULTS/'development'/'exp20_bounded_agent'; out.mkdir(parents=True, exist_ok=True)
(out/'predictions.jsonl').write_text('\n'.join(json.dumps(r) for r in recs)+'\n')
(out/'traces.json').write_text(json.dumps(results, indent=1, default=str))
(out/'summary.json').write_text(json.dumps(sA, indent=1, default=str))
exp18 = {json.loads(l)['case_id']: json.loads(l) for l in (C.RESULTS/'development'/'exp18_fixed_workflow'/'predictions.jsonl').read_text().splitlines()}
s18_c = sum(exp18[g['case_id']]['predicted_decision']==g['expected_decision'] for g in groupC)
print('Exp 20 agent on group C:', sA['correct'], '/', sA['n'], '| Exp 18 workflow on the same 13:', s18_c, '/ 13')
print('agent false approvals:', sA['false_approvals'], '/', sA['non_approvable'], '| human_review_rate', sA['human_review_rate'])
turns = [r['turns'] for r in results.values()]; import statistics as st
print('avg turns', round(st.mean(turns),2), '| step-cap hits', sum(r['step_cap_hit'] for r in results.values()), '| wrong-tool calls total', sum(r['wrong_tool_calls'] for r in results.values()))
print('total agent cost $%.4f' % sum(r['cost_usd'] for r in results.values()))

Exp 20 agent on group C: 7 / 13 | Exp 18 workflow on the same 13: 7 / 13
agent false approvals: 3 / 10 | human_review_rate 0.2308
avg turns 5.69 | step-cap hits 3 | wrong-tool calls total 20
total agent cost $0.0432


## Case-by-case: agent vs workflow vs expected

In [5]:
cmp_ = pd.DataFrame([{'case_id': g['case_id'], 'family': g['case_family'], 'expected': g['expected_decision'], 'agent': results[g['case_id']]['decision'], 'workflow_v2': exp18[g['case_id']]['predicted_decision'],
                     'agent_turns': results[g['case_id']]['turns'], 'agent_wrong_tool_calls': results[g['case_id']]['wrong_tool_calls']} for g in groupC])
cmp_

,case_id,family,expected,agent,workflow_v2,agent_turns,agent_wrong_tool_calls
0,X2-005,DYNAMIC_HOTEL_DISCOVERY,REQUEST_INFORMATION,REJECT,REQUEST_INFORMATION,6,2
1,X2-018,DYNAMIC_DELEGATION_CHAIN,REQUEST_INFORMATION,REJECT,REQUEST_INFORMATION,6,2
2,X2-032,DYNAMIC_DELEGATION_CHAIN,ESCALATE,REJECT,ESCALATE,5,1
3,X2-037,DYNAMIC_PROJECT_BUDGET_CHAIN,REQUEST_INFORMATION,APPROVE,ESCALATE,6,2
4,X2-047,DYNAMIC_DELEGATION_CHAIN,APPROVE,APPROVE,REQUEST_INFORMATION,5,1
5,X2-055,DYNAMIC_DELEGATION_CHAIN,APPROVE,APPROVE,REQUEST_INFORMATION,6,2
6,X2-087,DYNAMIC_HOTEL_DISCOVERY,ESCALATE,APPROVE,ESCALATE,4,0
7,X2-089,DYNAMIC_DEEP_HOTEL_CHAIN,APPROVE,APPROVE,APPROVE,6,1
8,X2-095,DYNAMIC_DEEP_HOTEL_CHAIN,REJECT,REJECT,REJECT,6,1
9,X2-104,DYNAMIC_HOTEL_DISCOVERY,ESCALATE,APPROVE,REQUEST_INFORMATION,6,1


## What was done and what it means
- **The agent ties the fixed workflow exactly: 7/13 each, on the same 13 group-C claims.** This is consistent with Exp 19's audit -- no case needed a model to invent a novel tool sequence -- but the agent does not clearly *lose* to the workflow either; it reaches the same accuracy through genuinely different (and imperfect) means.
- **The standout finding is complementary, not equal, errors.** The workflow's 7 correct cases and the agent's 7 correct cases overlap on only **3 claims** (X2-089, X2-095, X2-142). Between them they correctly solve **11 of 13 claims (85%)** -- each gets right several cases the other gets wrong. The two cases neither solves (X2-037, X2-104) are the two hardest in the set (a nested project-parent-budget chain and a hotel-exception case with a scope mismatch). **A system that could pick the better answer per case, or run both and reconcile, would materially outperform either alone on this group** -- direct empirical support for a selective architecture (Exp 30) rather than committing to one method for every dynamic case.
- **Safety is worse for the agent on this small subset:** agent false approvals 3/10 (30%) vs the workflow's 1/10 (10%) on the identical 10 non-approvable group-C claims -- well above the 10% guardrail. The agent's independent, model-directed reasoning is more prone to concluding APPROVE prematurely (e.g. X2-037, X2-087, X2-104 are all false approvals) than the workflow's conservative, pre-declared branches.
- **Orchestration mechanics show real inefficiency:** average 5.69 of 6 turns used (the agent rarely stops early even when it likely has enough evidence), 3 of 13 runs hit the step cap outright (all three still landed on ESCALATE, the safe fallback, and two of those three happened to be correct), and **20 wrong-tool-calls across 13 cases (about 1.5 per case)** -- calls that returned not-found, typically from guessing an identifier or calling a tool before establishing whether it was the right one. This is exactly the kind of orchestration overhead the plan's later tool-description and tool-confusion experiments (Exp 22-27) are designed to reduce, should an agent be kept in the architecture.
- **Cost:** $0.0432 for 13 claims (about $0.0033/claim, 5.7 turns average) -- cheap in absolute terms, but each claim needed roughly 6x the LLM calls of the single-shot hybrid systems in Exp 12 for no net accuracy gain over the workflow.
- **Decision, combining Exp 19 and Exp 20 as agreed:** Exp 19's audit found no case structurally requires agentic reasoning, and Exp 20's measured pilot confirms this -- the agent does not outperform a purely static workflow on accuracy, costs more (turns, wrong-tool calls, worse false-approval rate) to tie it, and only 3 of 13 cases are solved by both. **A bounded agent is not justified as a wholesale replacement for the fixed workflow on this dataset.** Per the plan's decision rule, Exp 21-27 (workflow vs. agent ablations, tool-set size, tool descriptions, parallelism, failure reproduction) are not warranted as a general architecture change -- but the complementary-error finding is worth carrying into Exp 30: a selective router that tries the workflow first and falls back to the agent (or vice versa) only on the workflow's known-weak families could recover several of the 6 currently-unique-to-one-method cases without paying the agent's cost on every claim. Next: Exp 28 (guardrail suite).